# Step 1 — Transit observables from a TTV posterior

**Goal.** Convert a light-curve/TTV fit posterior of $(R_p/R_\star,\ \rho_\star,\ b,\ P)$ into
the full set of transit observables $(\delta,\ a/R_\star,\ T_{14},\ T_{23},\ b,\ i_{\rm orb},\ P)$
that the ring-geometry likelihood (steps 2–3) consumes.

This notebook is a **thin guide**: the actual computation lives in
[`photoring.observables`](photoring/observables.py) (`derive_observables`), so it is identical
across every case. Here we only choose the *case*, point at its input files, and run it.

### Two ways to feed the pipeline

| Entry | You provide | This notebook |
|---|---|---|
| **A. Raw TTV posterior** | MultiNest-style `post_equal_weights.dat` with columns `(Rp/R*, ρ*[kg/m³], b, P[days], …, logL)` | derives the observables and writes `<case>_<planet>_observables.dat` |
| **B. Pre-derived observables** | a `.dat` already in the 9-column observables format (see below) | **skip this notebook** and go straight to step 2 |

The derived-observables file has columns:
`p, δ, a/R★, ρ_obs[kg/m³], P[days], b, i_orb[°], T14[h], T23[h]`.

### The math (Zuluaga et al. 2015 / Kipping 2014, Eq. 1–4)

$$\delta = p^2,\qquad \frac{a}{R_\star}=\Big(\frac{G\,\rho_\star\,P^2}{3\pi}\Big)^{1/3},$$
$$T_{14/23}=\frac{P}{\pi}\,\arcsin\!\Big[\sqrt{\tfrac{(1\pm p)^2-b^2}{(a/R_\star)^2-b^2}}\Big],\qquad
i_{\rm orb}=\arccos\!\big(b/(a/R_\star)\big).$$

These are **deterministic transforms** of the posterior columns — the observable samples inherit
the full covariance of the photometric fit, so no error propagation is needed.

## 0. Environment

In [ ]:
# ── Bootstrap: make the sibling packages importable without installation ────
# The pipeline uses three packages that live in the repository, uninstalled:
#   exorings, geotrans   (repo root)      photoring   (pipeline/)
# We locate the repo root and pipeline/ robustly from the current working dir
# (Jupyter / nbconvert / papermill all run notebooks from pipeline/).
import sys, pathlib
_HERE   = pathlib.Path.cwd().resolve()
_cands  = [_HERE, *_HERE.parents]
_NB_DIR = next((c for c in _cands if (c / "photoring").is_dir()), _HERE)
_REPO   = next((c for c in _cands if (c / "exorings").is_dir()), _NB_DIR.parent)
for _p in (str(_REPO), str(_NB_DIR)):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("repo root :", _REPO)
print("pipeline  :", _NB_DIR)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
import warnings; warnings.filterwarnings("ignore")

import photoring as pr
from photoring.observables import load_posterior, derive_observables, save_observables
import photoring.plotting as plot
plot.apply_style()
print("photoring ready.")

## 1. Case configuration  ← edit here for your target

`CASE` selects the case directory `pipeline/<CASE>/`. `PLANET_TTV` lists the raw TTV file(s)
for each planet, **relative to** `pipeline/<CASE>/inputs/ttv/`. A planet may have several
light-curve segments (Kepler-51 b has three); list them all.

To run your **own** target: copy `pipeline/kepler_51/` to `pipeline/<your_case>/`, drop your TTV
files under `inputs/ttv/`, set `CASE` and `PLANET_TTV` below, and adjust `COLS` to your columns.

In [ ]:
CASE = "kepler_51"
paths = pr.CasePaths(CASE)

# Raw TTV posterior file(s) per planet, relative to inputs/ttv/.
PLANET_TTV = {
    "b": ["Kepler-51b/TTVplan1-post_equal_weights.dat",
          "Kepler-51b/TTVplan2-post_equal_weights.dat",
          "Kepler-51b/TTVplan3-post_equal_weights.dat"],
    "d": ["Kepler-51d/TTVplan-post_equal_weights.dat"],
}

# Column indices in the TTV files: Rp/R*, rho*[kg/m3], b, P[days], t_mid, ..., logL(last).
COLS = dict(col_p=0, col_rho=1, col_b=2, col_per=3, col_tmid=4, col_logl=-1)

# Segment-merge policy when a planet has several segments (see markdown in §2).
#   "last" -> use the most complete segment (Kepler-51 default)
#   "pool" -> concatenate all segments
SEGMENT_MERGE = "last"
print("case:", CASE, "| planets:", list(PLANET_TTV))

## 2. Derive and save observables

For a single-segment planet the derivation is direct. For a **multi-segment** planet
(Kepler-51 b) the segments are independent posteriors over the same parameters; we check their
mutual consistency with two-sample KS tests and, by default, adopt the most complete segment as
the reference posterior (`SEGMENT_MERGE="last"`). Pooling all segments (`"pool"`) is available as
a more conservative alternative.

In [ ]:
obs_by_planet = {}
for planet, files in PLANET_TTV.items():
    segs = [derive_observables(load_posterior(paths.ttv_dir / f, **COLS)) for f in files]

    if len(segs) == 1:
        obs_final = segs[0].dropna()
    elif SEGMENT_MERGE == "pool":
        obs_final = pd.concat(segs, ignore_index=True).dropna()
    else:  # "last": most complete segment
        obs_final = segs[-1].dropna()

    # Consistency check across segments (informative only)
    if len(segs) > 1:
        print(f"planet {planet}: KS consistency across segments")
        for key in ("T14", "T23", "delta"):
            for i in range(len(segs)):
                for j in range(i + 1, len(segs)):
                    vi = segs[i][key].dropna().values; vj = segs[j][key].dropna().values
                    st, pv = ks_2samp(vi, vj)
                    flag = "" if pv > 0.05 else "  <- differ"
                    print(f"    {key:5s} seg{i+1} vs seg{j+1}: KS={st:.3f} p={pv:.3f}{flag}")

    obs_by_planet[planet] = obs_final
    out = save_observables(obs_final, paths.observables_file(planet))
    print(f"planet {planet}: {len(obs_final)} samples -> {paths.observables_file(planet).name}  {out.shape}")

## 3. Posterior summary and figure

In [ ]:
def summary(obs, planet):
    print(f"\n=== Planet {planet} — observable posterior summary ===")
    for key in ["p", "delta", "aR", "P", "b", "i_orb", "T14", "T23"]:
        v = obs[key].dropna().values
        print(f"  {key:>6}:  mean={v.mean():>10.5f}  std={v.std():>9.5f}  median={np.median(v):>10.5f}")

for planet, obs in obs_by_planet.items():
    summary(obs, planet)

In [ ]:
from photoring.io import load_observables
ttv_by_planet = {pl: load_observables(paths.observables_file(pl)) for pl in PLANET_TTV}
plot.plot_observable_posteriors(ttv_by_planet, paths=paths, run_tag=f"{CASE}_observables")

## 4. Optional diagnostics — ring-stability context (Kepler-51)

The following is **case-specific** commentary, not part of the core pipeline: it compares each
planet's Hill radius (dynamical stability outer bound) and Roche radius (tidal inner bound for
ring material) to gauge whether rings could plausibly survive. Adapt the stellar/planet masses
and radii for your own target, or delete this section.

In [ ]:
import astropy.constants as c
import astropy.units as u

# Kepler-51 system properties (Masuda 2014; Libby-Roberts 2020)
M_s, R_s = 0.96 * c.M_sun, 0.87 * c.R_sun
props = {  # planet: (mass, radius)
    "b": (6.7 * c.M_earth, 6.83 * c.R_earth),
    "d": (5.6 * c.M_earth, 9.32 * c.R_earth),
}
rho_ring = {"icy": 1.0e3 * u.kg / u.m**3, "dusty": 3.0e3 * u.kg / u.m**3}

for planet, (m_p, R_p) in props.items():
    if planet not in obs_by_planet:
        continue
    aR = obs_by_planet[planet]["aR"].dropna().values.mean()
    R_H = aR * R_s * (m_p / (3 * M_s)) ** (1 / 3)                 # Hill radius
    rho_p = (m_p / (4 / 3 * np.pi * R_p ** 3)).to(u.kg / u.m**3)  # planet bulk density
    print(f"\nPlanet {planet}:  R_Hill = {float((R_H / R_p).decompose()):.1f} Rp")
    for kind, rho_r in rho_ring.items():
        R_roche = R_p * (2 * rho_p / rho_r) ** (1 / 3)            # Roche radius (fluid)
        print(f"    R_Roche ({kind:5s} rings) = {float((R_roche / R_p).decompose()):.2f} Rp")

---
**Next:** step 2 builds the KDE likelihood from these observables and samples the ring geometry —
[`02_inference_dynesty.ipynb`](02_inference_dynesty.ipynb) (nested sampling, gives $\ln\mathcal{Z}$)
or [`02_inference_emcee.ipynb`](02_inference_emcee.ipynb) (MCMC).